# 3.3a MLOps: pipelines, model registry, safe deployments and CI/CD

**Story:** Until now you trained and deployed the SafeDrive fraud model by hand. New claims arrive every day and
fraud patterns change, so the model must be rebuilt, tested, approved and rolled out the same way every time,
with a record of every version for the auditors.

**Task 3.3:** Implement automated orchestration and CI/CD pipelines for MLOps and AI workloads.

| Skill | What you do here | Status |
|---|---|---|
| 3.3.3 | SageMaker pipeline: processing, training, evaluation, condition, register | ✅ |
| 3.3.6 | Model Registry: versions, approval, deploy the approved version; MLflow note | ✅ |
| 3.3.1 | Blue/green canary deployment with alarm-based auto-rollback; linear, all-at-once, rolling | ✅ |
| 3.3.2 | CodeBuild and CodePipeline created; CodeCommit, CodeDeploy, CodeConnections notes; troubleshooting | ✅ |
| 3.3.4 | Tests in CI/CD: unit test, model-quality gate, endpoint smoke test | ✅ |
| 3.3.5 | Retraining triggers with EventBridge: schedule, new data in S3, drift alarm | ✅ |

Run time is about 60 minutes (the pipeline run and the canary deployment are mostly waiting).

## Key points for the exam

- **SageMaker Pipelines** = a DAG of steps (Processing, Training, Tuning, Transform, Model, Condition, Fail, Lambda, Callback ...).
  Steps are linked by **properties** of earlier steps. **Parameters** change a run without changing the code.
- A **PropertyFile** + `JsonGet` lets a `ConditionStep` read a number (for example AUC) from a JSON file a step wrote.
- **Step caching** skips a step when its inputs did not change.
- **Model Registry:** model package **group** holds numbered **versions**. Each has an approval status:
  `PendingManualApproval`, `Approved`, `Rejected`. Approval can trigger deployment through EventBridge.
- **Deployment guardrails:** blue/green with `ALL_AT_ONCE`, `CANARY` or `LINEAR` traffic shifting, a baking period,
  and **auto-rollback** when a CloudWatch alarm fires. **Rolling** updates batches in place (less extra capacity).
- **CodePipeline** orchestrates stages; **CodeBuild** runs commands from `buildspec.yml`; **CodeDeploy** shifts traffic for
  EC2, ECS and Lambda; **CodeConnections** links GitHub, GitLab or Bitbucket.
- **Retraining** is started by EventBridge: a schedule, an S3 "Object Created" event, or a Model Monitor alarm.

| Traffic shifting | What happens | Choose it when |
|---|---|---|
| All at once | 100% moves to the new fleet, then baking | Fast, low-risk changes |
| Canary | A small part first, then the rest | Limit the blast radius, quick rollout |
| Linear | Equal steps (for example 25% every 10 minutes) | Most cautious, slowest |
| Rolling | Replace instances batch by batch | Large fleets, limited quota for a second fleet |

## Setup

In [ ]:
import sys; sys.path.append("..")
import io, json, os, shutil, time, zipfile
import pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
os.makedirs("scripts/ci/tests", exist_ok=True)
print(U.REGION, U.bucket())

**Catch-up:** raw data in S3 and the serverless endpoint `safedrive-fraud` (used by the smoke test).
Both return at once if earlier notebooks built them.

In [ ]:
RAW = C.ensure_raw_data()
ENDPOINT = C.ensure_fraud_endpoint()
state = U.load_state()
ROLE, BUCKET, REGION = U.role_arn(), U.bucket(), U.REGION
print(RAW, ENDPOINT)

Clients and names used in this notebook.

In [ ]:
sm, smr, s3, cw = U.client("sagemaker"), U.client("sagemaker-runtime"), U.client("s3"), U.client("cloudwatch")
PIPELINE_NAME = "safedrive-fraud-pipeline"
GROUP = "safedrive-fraud"
PREFIX = "safedrive/pipeline"

def exists(fn, **kwargs):
    try:
        fn(**kwargs)
        return True
    except ClientError:
        return False

print(PIPELINE_NAME, GROUP)

## 3.3.3 Configure training and inference jobs as a SageMaker pipeline

- Each step is a normal SageMaker job (processing, training ...) with its own image, instance type and inputs.
- With a `PipelineSession`, calls like `.run()` and `.fit()` do **not** start a job. They return the job
  settings, which become a pipeline step.
- The pipeline we build:
  `BuildFeatures` (processing) → `TrainXGBoost` (training) → `EvaluateModel` (processing) →
  `CheckAUC` (condition) → `RegisterFraudModel` or `AucTooLow` (fail).

Create the pipeline session and the **parameters** (values you can change per run).

In [ ]:
import sagemaker
from sagemaker.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.workflow.pipeline_context import PipelineSession
psess = PipelineSession(boto_session=U.session, default_bucket=BUCKET)

claims_data = ParameterString(name="ClaimsData", default_value=U.s3_uri(f"{RAW}/claims.csv"))
auc_threshold = ParameterFloat(name="AucThreshold", default_value=0.75)
approval_status = ParameterString(name="ApprovalStatus", default_value="PendingManualApproval")
print([p.name for p in (claims_data, auc_threshold, approval_status)])

### Step 1: feature build (ProcessingStep)
The script reads the raw claims and policies, builds the features with the same `features.py` used in notebook 1.2a,
and writes train, validation and test CSV files. A processing job sees its inputs and outputs under `/opt/ml/processing`.

In [ ]:
%%writefile scripts/preprocess.py
"""Pipeline step 1: build features and split into train / validation / test."""
import glob
import os
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

BASE = os.environ.get("PROCESSING_DIR", "/opt/ml/processing")
sys.path.append(f"{BASE}/lib")          # features.py arrives here as a processing input
import features as F  # noqa: E402

claims = pd.read_csv(glob.glob(f"{BASE}/input/claims/*.csv")[0])
policies = pd.read_csv(glob.glob(f"{BASE}/input/policies/*.csv")[0])
data = F.build_features(claims, policies).drop(columns="claim_id")

train, rest = train_test_split(data, test_size=0.3, stratify=data[F.LABEL], random_state=42)
val, test = train_test_split(rest, test_size=0.5, stratify=rest[F.LABEL], random_state=42)

for name, part in [("train", train), ("validation", val), ("test", test)]:
    os.makedirs(f"{BASE}/{name}", exist_ok=True)
    part.to_csv(f"{BASE}/{name}/{name}.csv", index=False, header=False)   # label first, no header
    print(name, part.shape, "fraud rate", round(part[F.LABEL].mean(), 4))

Configure the processing job: scikit-learn image, one `ml.m5.large`. `cache` lets the pipeline skip steps whose inputs did not change.

In [ ]:
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.steps import CacheConfig, ProcessingStep, TrainingStep
cache = CacheConfig(enable_caching=True, expire_after="P7D")

sk_processor = SKLearnProcessor(framework_version="1.2-1", role=ROLE, instance_type="ml.m5.large",
                                instance_count=1, base_job_name="safedrive-features",
                                sagemaker_session=psess, tags=U.TAGS)
print(sk_processor.image_uri)

Declare inputs (S3 → container folder) and outputs (container folder → S3). Because of the pipeline session, `run()` only returns the settings.

In [ ]:
P = "/opt/ml/processing"
process_args = sk_processor.run(
    code="scripts/preprocess.py",
    inputs=[ProcessingInput(source=claims_data, destination=f"{P}/input/claims"),
            ProcessingInput(source=U.s3_uri(f"{RAW}/policies.csv"), destination=f"{P}/input/policies"),
            ProcessingInput(source=os.path.join(U.REPO_ROOT, "common", "features.py"), destination=f"{P}/lib")],
    outputs=[ProcessingOutput(output_name=n, source=f"{P}/{n}") for n in ["train", "validation", "test"]])
step_process = ProcessingStep(name="BuildFeatures", step_args=process_args, cache_config=cache)
print(step_process.name, step_process.step_type)

### Step 2: training (TrainingStep)
The built-in XGBoost container, same hyperparameters as notebook 2.2a. The inputs are **properties of step 1**,
which is what creates the dependency between the steps.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
XGB_IMAGE = sagemaker.image_uris.retrieve("xgboost", REGION, version="1.7-1")
estimator = Estimator(image_uri=XGB_IMAGE, role=ROLE, instance_count=1, instance_type="ml.m5.large",
                      output_path=U.s3_uri(f"{PREFIX}/models"), sagemaker_session=psess,
                      base_job_name="safedrive-fraud-pipeline", tags=U.TAGS)
estimator.set_hyperparameters(objective="binary:logistic", eval_metric="auc", num_round=150,
                              max_depth=5, eta=0.1, subsample=0.8, scale_pos_weight=10)
print(XGB_IMAGE)

In [ ]:
outputs = step_process.properties.ProcessingOutputConfig.Outputs
train_args = estimator.fit({
    "train": TrainingInput(outputs["train"].S3Output.S3Uri, content_type="text/csv"),
    "validation": TrainingInput(outputs["validation"].S3Output.S3Uri, content_type="text/csv")})
step_train = TrainingStep(name="TrainXGBoost", step_args=train_args, cache_config=cache)
print(step_train.name, step_train.step_type)

### Step 3: evaluation (ProcessingStep with a PropertyFile)
The script loads the trained model, scores the test set and writes `evaluation.json`.

In [ ]:
%%writefile scripts/evaluate.py
"""Pipeline step 3: score the test set and write evaluation.json (read by the ConditionStep)."""
import json
import os
import tarfile

import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = os.environ.get("PROCESSING_DIR", "/opt/ml/processing")

with tarfile.open(f"{BASE}/model/model.tar.gz") as tar:
    tar.extractall(f"{BASE}/model")
booster = xgb.Booster()
booster.load_model(f"{BASE}/model/xgboost-model")

test = pd.read_csv(f"{BASE}/test/test.csv", header=None)
y, X = test.iloc[:, 0], test.iloc[:, 1:]
scores = booster.predict(xgb.DMatrix(X.values))

report = {"binary_classification_metrics": {
    "auc": {"value": float(roc_auc_score(y, scores))},
    "pr_auc": {"value": float(average_precision_score(y, scores))}}}
os.makedirs(f"{BASE}/evaluation", exist_ok=True)
with open(f"{BASE}/evaluation/evaluation.json", "w") as f:
    json.dump(report, f)
print(report)

The evaluation job runs in the XGBoost image (it has `xgboost`, `pandas` and `scikit-learn`).
The model artifact of step 2 and the test set of step 1 are its inputs.

In [ ]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile
eval_processor = ScriptProcessor(image_uri=XGB_IMAGE, command=["python3"], role=ROLE, instance_count=1,
                                 instance_type="ml.m5.large", base_job_name="safedrive-evaluate",
                                 sagemaker_session=psess, tags=U.TAGS)
eval_args = eval_processor.run(
    code="scripts/evaluate.py",
    inputs=[ProcessingInput(source=step_train.properties.ModelArtifacts.S3ModelArtifacts, destination=f"{P}/model"),
            ProcessingInput(source=outputs["test"].S3Output.S3Uri, destination=f"{P}/test")],
    outputs=[ProcessingOutput(output_name="evaluation", source=f"{P}/evaluation")])
print("evaluation job configured")

The **PropertyFile** tells the pipeline that the output named `evaluation` contains `evaluation.json`, so later steps can read values from it.

In [ ]:
evaluation_report = PropertyFile(name="EvaluationReport", output_name="evaluation", path="evaluation.json")
step_eval = ProcessingStep(name="EvaluateModel", step_args=eval_args,
                           property_files=[evaluation_report], cache_config=cache)
print(step_eval.name, step_eval.step_type)

### Step 4: register the model (ModelStep)
First make sure the **model package group** exists. It is the folder that holds all versions of the fraud model.

In [ ]:
if not exists(sm.describe_model_package_group, ModelPackageGroupName=GROUP):
    sm.create_model_package_group(ModelPackageGroupName=GROUP, Tags=U.TAGS,
                                  ModelPackageGroupDescription="SafeDrive claim-fraud model versions")
sm.describe_model_package_group(ModelPackageGroupName=GROUP)["ModelPackageGroupStatus"]

The registered version stores the image, the artifact, the instance types it may run on, the approval status
and a link to the evaluation report (`ModelMetrics`). `model.register(...)` in a `ModelStep` is the current form of the older `RegisterModel` step.

In [ ]:
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.functions import Join, JsonGet
from sagemaker.workflow.model_step import ModelStep
eval_uri = step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri
metrics = ModelMetrics(model_statistics=MetricsSource(
    s3_uri=Join(on="/", values=[eval_uri, "evaluation.json"]), content_type="application/json"))
model = Model(image_uri=XGB_IMAGE, model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
              role=ROLE, sagemaker_session=psess)
register_args = model.register(
    content_types=["text/csv"], response_types=["text/csv"], inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"], model_package_group_name=GROUP,
    approval_status=approval_status, model_metrics=metrics)
step_register = ModelStep(name="RegisterFraudModel", step_args=register_args)
print(step_register.name)

### Step 5: quality gate (ConditionStep)
`JsonGet` reads the AUC from the property file. If it is at least the threshold, register the model; otherwise the pipeline fails with a clear message.

In [ ]:
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
auc = JsonGet(step_name=step_eval.name, property_file=evaluation_report,
              json_path="binary_classification_metrics.auc.value")
step_fail = FailStep(name="AucTooLow",
                     error_message=Join(on=" ", values=["AUC is below the threshold", auc_threshold]))
step_cond = ConditionStep(name="CheckAUC",
                          conditions=[ConditionGreaterThanOrEqualTo(left=auc, right=auc_threshold)],
                          if_steps=[step_register], else_steps=[step_fail])
print(step_cond.name, step_cond.step_type)

### Create the pipeline
`upsert` creates the pipeline, or updates it if it exists. Only the steps are listed; the order comes from their dependencies.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
pipeline = Pipeline(name=PIPELINE_NAME, parameters=[claims_data, auc_threshold, approval_status],
                    steps=[step_process, step_train, step_eval, step_cond], sagemaker_session=psess)
pipeline_arn = pipeline.upsert(role_arn=ROLE, tags=U.TAGS)["PipelineArn"]
print(pipeline_arn)

The pipeline is stored as a JSON definition. Look at the steps and their types.

In [ ]:
definition = json.loads(pipeline.definition())
print("parameters:", [p["Name"] for p in definition["Parameters"]])
[(s["Name"], s["Type"]) for s in definition["Steps"]]

### Run it
Start an execution with the default parameters.

In [ ]:
execution = pipeline.start(execution_display_name="first-run")
EXECUTION_ARN = execution.arn
print(EXECUTION_ARN)

Wait for the execution (about 12 to 15 minutes: three jobs, each starts its own instance).
You can also watch the graph in SageMaker Studio under Pipelines.

In [ ]:
def execution_status(arn):
    return sm.describe_pipeline_execution(PipelineExecutionArn=arn)["PipelineExecutionStatus"]

U.wait_until(lambda: execution_status(EXECUTION_ARN), ok=("Succeeded",), bad=("Failed", "Stopped"),
             delay=30, timeout=2700, label="pipeline execution")

List the steps of the execution: status, and the job each step started.

In [ ]:
def step_table(arn):
    steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=arn)["PipelineExecutionSteps"]
    rows = [{"step": s["StepName"], "status": s["StepStatus"], "start": s.get("StartTime"),
             "cache_hit": "CacheHitResult" in s, "details": json.dumps(s.get("Metadata", {}))[:90]}
            for s in steps]
    return pd.DataFrame(rows).sort_values("start")

step_table(EXECUTION_ARN)

## 3.3.6 Manage model versions for repeatability and audits

- **SageMaker Model Registry:** every pipeline run that passes the gate adds a new **version** to the group, with the
  artifact, image, metrics, approval status, who approved it and lineage back to the training job and data.
- Deploy **from the registry** (by model package ARN), not from a loose S3 path. That makes a deployment repeatable.
- Version status changes emit EventBridge events, so "Approved" can start a deployment pipeline.

List the versions in the group.

In [ ]:
packages = sm.list_model_packages(ModelPackageGroupName=GROUP, SortBy="CreationTime",
                                  SortOrder="Descending")["ModelPackageSummaryList"]
pd.DataFrame(packages)[["ModelPackageVersion", "ModelApprovalStatus", "ModelPackageStatus", "CreationTime"]]

Look inside the newest version: where the artifact is, and the evaluation report that was stored with it.

In [ ]:
LATEST_ARN = packages[0]["ModelPackageArn"]
package = sm.describe_model_package(ModelPackageName=LATEST_ARN)
metrics_uri = package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
print(package["InferenceSpecification"]["Containers"][0]["ModelDataUrl"])
print(json.loads(U.read_bytes(metrics_uri.split(f"{BUCKET}/", 1)[1])))

Approve the version. In a real team a reviewer does this in Studio after reading the metrics.

In [ ]:
sm.update_model_package(ModelPackageArn=LATEST_ARN, ModelApprovalStatus="Approved",
                        ApprovalDescription="AUC above threshold, reviewed in notebook 3.3a")
sm.describe_model_package(ModelPackageName=LATEST_ARN)["ModelApprovalStatus"]

Deploy the latest **approved** version. Find it by filtering on the approval status.

In [ ]:
approved = sm.list_model_packages(ModelPackageGroupName=GROUP, ModelApprovalStatus="Approved",
                                  SortBy="CreationTime", SortOrder="Descending")["ModelPackageSummaryList"][0]
BLUE = f"safedrive-fraud-v{approved['ModelPackageVersion']}"
if not exists(sm.describe_model, ModelName=BLUE):
    sm.create_model(ModelName=BLUE, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    Containers=[{"ModelPackageName": approved["ModelPackageArn"]}])
print(BLUE, "<-", approved["ModelPackageArn"])

Endpoint config and a real-time endpoint `safedrive-fraud-prod`. The canary deployment in the next section needs
at least 2 instances. 💰 About $0.23 per hour until deleted.
If your quota for `ml.m5.large` endpoint instances is below 4, set `INSTANCES = 1` here.

In [ ]:
PROD, INSTANCES = "safedrive-fraud-prod", 2
if not exists(sm.describe_endpoint_config, EndpointConfigName=BLUE):
    sm.create_endpoint_config(
        EndpointConfigName=BLUE, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": BLUE,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": INSTANCES}])
if not exists(sm.describe_endpoint, EndpointName=PROD):
    sm.create_endpoint(EndpointName=PROD, EndpointConfigName=BLUE, Tags=U.TAGS)
print(PROD, "->", sm.describe_endpoint(EndpointName=PROD)["EndpointStatus"])

Wait for the endpoint (about 5 minutes) and test it with one claim from the test set.

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=PROD)
TEST_KEY = C.ensure_training_data()["test_s3"].split(f"{BUCKET}/", 1)[1]
payload = U.read_csv(TEST_KEY, header=None).iloc[:3, 1:].to_csv(header=False, index=False).strip()
resp = smr.invoke_endpoint(EndpointName=PROD, ContentType="text/csv", Body=payload)
print(resp["Body"].read().decode().split())

**MLflow on SageMaker AI (note):** a managed MLflow tracking server (or the serverless MLflow app) stores experiments,
runs and its own model registry. A model registered in MLflow is **also registered in SageMaker Model Registry automatically**,
so approval and deployment work the same way.

```python
import mlflow
mlflow.set_tracking_uri(tracking_server_arn)            # needs the sagemaker-mlflow plugin
with mlflow.start_run():
    mlflow.log_metric("auc", 0.83)
    mlflow.xgboost.log_model(booster, "model", registered_model_name="safedrive-fraud")
```

| Need | Use |
|---|---|
| Approval workflow, deployment from a version, cross-account sharing | SageMaker Model Registry |
| Experiment tracking, comparing runs, open-source tooling | MLflow on SageMaker AI |
| Who trained what from which data | SageMaker lineage tracking, Model Cards |

## 3.3.1 Automated deployment strategies and rollback

- `UpdateEndpoint` with a new endpoint config is a **blue/green** deployment: a new (green) fleet is created next to the old (blue) one.
- `DeploymentConfig` adds guardrails: **traffic routing** (`ALL_AT_ONCE`, `CANARY`, `LINEAR`), a **baking period**
  (`WaitIntervalInSeconds`) and **auto-rollback alarms**. If an alarm fires during baking, traffic goes back to blue.
- `TerminationWaitInSeconds` keeps the blue fleet for a while after the shift, for a fast rollback.
- **Rolling** (`RollingUpdatePolicy`) updates in batches without a full second fleet.

First create the alarm that guards the deployment: any 5XX error on the endpoint.

In [ ]:
ROLLBACK_ALARM = "safedrive-fraud-prod-5xx"
cw.put_metric_alarm(
    AlarmName=ROLLBACK_ALARM, Namespace="AWS/SageMaker", MetricName="Invocation5XXErrors",
    Dimensions=[{"Name": "EndpointName", "Value": PROD}, {"Name": "VariantName", "Value": "AllTraffic"}],
    Statistic="Sum", Period=60, EvaluationPeriods=1, Threshold=1,
    ComparisonOperator="GreaterThanOrEqualToThreshold", TreatMissingData="notBreaching", Tags=U.TAGS)
cw.describe_alarms(AlarmNames=[ROLLBACK_ALARM])["MetricAlarms"][0]["StateValue"]

Create the new (green) model and endpoint config. We use the tuned model from notebook 2.2a if it exists,
otherwise the same model package again (the deployment mechanics are identical).

In [ ]:
GREEN = f"{BLUE}-green"
v2 = state.get("fraud_model_data_v2")
container = {"Image": XGB_IMAGE, "ModelDataUrl": v2} if v2 else {"ModelPackageName": approved["ModelPackageArn"]}
if not exists(sm.describe_model, ModelName=GREEN):
    sm.create_model(ModelName=GREEN, ExecutionRoleArn=ROLE, Tags=U.TAGS, Containers=[container])
if not exists(sm.describe_endpoint_config, EndpointConfigName=GREEN):
    sm.create_endpoint_config(
        EndpointConfigName=GREEN, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": GREEN,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": INSTANCES}])
print("green config:", GREEN)

The traffic routing: **canary** = send traffic to 1 green instance first, bake for 2 minutes, then shift the rest.
With `INSTANCES = 1` a canary is not possible, so the code falls back to all-at-once.

In [ ]:
routing = {"Type": "CANARY", "CanarySize": {"Type": "INSTANCE_COUNT", "Value": 1}, "WaitIntervalInSeconds": 120}
if INSTANCES < 2:
    routing = {"Type": "ALL_AT_ONCE", "WaitIntervalInSeconds": 120}
routing

Start the deployment. Watch for `Updating` in the output.

In [ ]:
sm.update_endpoint(
    EndpointName=PROD, EndpointConfigName=GREEN,
    DeploymentConfig={
        "BlueGreenUpdatePolicy": {"TrafficRoutingConfiguration": routing,
                                  "TerminationWaitInSeconds": 60,
                                  "MaximumExecutionTimeoutInSeconds": 1800},
        "AutoRollbackConfiguration": {"Alarms": [{"AlarmName": ROLLBACK_ALARM}]}})
print(PROD, "->", sm.describe_endpoint(EndpointName=PROD)["EndpointStatus"])

While it runs, look at the pending deployment: the green fleet is being prepared next to the blue one.

In [ ]:
pending = sm.describe_endpoint(EndpointName=PROD).get("PendingDeploymentSummary", {})
print("target config:", pending.get("EndpointConfigName"))
pending.get("ProductionVariants")

Wait until the endpoint is `InService` again (about 8 to 10 minutes: start green, canary, bake, shift, bake, remove blue).
The endpoint keeps answering requests the whole time.

In [ ]:
U.wait_until(lambda: sm.describe_endpoint(EndpointName=PROD)["EndpointStatus"], ok=("InService",),
             bad=("Failed", "UpdateRollbackFailed"), delay=30, timeout=2400, label="endpoint")

Check the result. The endpoint now uses the green config. If the alarm had fired, it would still show the blue
config and `FailureReason` would explain the rollback.

In [ ]:
ep = sm.describe_endpoint(EndpointName=PROD)
print("config now  :", ep["EndpointConfigName"], "(green)" if ep["EndpointConfigName"] == GREEN else "(rolled back)")
print("failure     :", ep.get("FailureReason"))
print("last routing:", ep.get("LastDeploymentConfig", {}).get("BlueGreenUpdatePolicy", {}).get("TrafficRoutingConfiguration"))

**Try a rollback yourself (optional):** start another `update_endpoint` back to the blue config with the same
`DeploymentConfig`, and while the status is `Updating` force the alarm:

```python
cw.set_alarm_state(AlarmName=ROLLBACK_ALARM, StateValue="ALARM", StateReason="rollback test")
```

Other policies use the same call, only the `DeploymentConfig` changes:

```python
# Linear: 50% of capacity per step, 5 minutes between steps
{"Type": "LINEAR", "LinearStepSize": {"Type": "CAPACITY_PERCENT", "Value": 50}, "WaitIntervalInSeconds": 300}
# Rolling: replace 1 instance at a time, no full second fleet
{"RollingUpdatePolicy": {"MaximumBatchSize": {"Type": "INSTANCE_COUNT", "Value": 1}, "WaitIntervalInSeconds": 120},
 "AutoRollbackConfiguration": {"Alarms": [{"AlarmName": ROLLBACK_ALARM}]}}
```

Other safe-rollout tools: **production variants** with weights (A/B test), **shadow variants** (copy of live traffic,
responses not returned), and for Lambda or ECS targets **CodeDeploy** canary and linear deployments.

Delete the production endpoint now (💰 two instances).

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=PROD)
U.safe(cw.delete_alarms, AlarmNames=[ROLLBACK_ALARM])

## 3.3.4 Automated tests in a CI/CD pipeline

| Test | What it checks | Needs AWS? |
|---|---|---|
| Unit test | Feature code gives the right columns and values | No |
| Model-quality gate | The newest registered model has AUC above the threshold | Model Registry, S3 |
| Endpoint smoke test | The deployed endpoint answers with a valid probability | Endpoint |
| (also common) data validation, bias checks, integration and load tests | | |

For GenAI workloads the same idea applies with prompt regression tests (notebook 3.3b).

The "repository" is the folder `scripts/ci`. Copy the feature code into it, then write the tests.

In [ ]:
shutil.copy(os.path.join(U.REPO_ROOT, "common", "features.py"), "scripts/ci/features.py")
os.listdir("scripts/ci")

Unit test: a tiny hand-made claim and policy, no AWS calls. It protects the feature contract between training and inference.

In [ ]:
%%writefile scripts/ci/tests/test_features.py
"""Unit test of the feature code (no AWS needed)."""
import pandas as pd

import features as F

CLAIMS = pd.DataFrame([
    {"claim_id": "C1", "policy_id": "P1", "claim_date": "2025-03-01", "report_delay_days": -2,
     "claim_type": " Theft ", "claim_amount": 5000.0, "incident_hour": 2, "police_report": "N",
     "witnesses": 0, "num_vehicles": 1, "channel": "app", "is_fraud": 1}])
POLICIES = pd.DataFrame([
    {"policy_id": "P1", "age": 22, "region": "North", "vehicle_year": 2020, "vehicle_value": 20000.0,
     "annual_premium": 1000.0, "tenure_months": 3, "prior_claims": 2}])


def test_columns_match_the_model_contract():
    out = F.build_features(CLAIMS, POLICIES)
    assert list(out.columns) == ["claim_id", F.LABEL] + F.FEATURES
    assert not out.isna().any().any()


def test_feature_values():
    row = F.build_features(CLAIMS, POLICIES).iloc[0]
    assert row["report_delay_days"] == 0            # negative delay is fixed
    assert row["claim_type_theft"] == 1             # text is cleaned before encoding
    assert row["night_incident"] == 1 and row["young_driver"] == 1 and row["new_customer"] == 1
    assert row["claim_to_value_ratio"] == 0.25

Run the unit test locally first (fast feedback before anything is pushed).

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_features.py"],
                        cwd="scripts/ci", capture_output=True, text=True)
print(result.stdout[-600:], result.stderr[-300:])

Model-quality gate: read the evaluation report of the newest version in the Model Registry and compare the AUC with a threshold.

In [ ]:
%%writefile scripts/ci/tests/test_model_quality.py
"""Model-quality gate: the newest registered model must reach the AUC threshold."""
import json
import os

import boto3


def test_latest_model_meets_auc_threshold():
    sagemaker_client, s3_client = boto3.client("sagemaker"), boto3.client("s3")
    latest = sagemaker_client.list_model_packages(
        ModelPackageGroupName=os.environ["MODEL_PACKAGE_GROUP"], SortBy="CreationTime",
        SortOrder="Descending", MaxResults=1)["ModelPackageSummaryList"][0]
    package = sagemaker_client.describe_model_package(ModelPackageName=latest["ModelPackageArn"])
    uri = package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
    bucket, key = uri.replace("s3://", "").split("/", 1)
    report = json.loads(s3_client.get_object(Bucket=bucket, Key=key)["Body"].read())
    auc = report["binary_classification_metrics"]["auc"]["value"]
    print("AUC of the latest model:", auc)
    assert auc >= float(os.environ.get("AUC_THRESHOLD", "0.75"))

Endpoint smoke test: send one claim to the endpoint and check that the answer is a probability.

In [ ]:
%%writefile scripts/ci/tests/test_endpoint_smoke.py
"""Smoke test: the deployed endpoint answers with a probability between 0 and 1."""
import os

import boto3

import features as F


def test_endpoint_returns_a_probability():
    runtime = boto3.client("sagemaker-runtime")
    row = ",".join(["0.5"] * len(F.FEATURES))
    resp = runtime.invoke_endpoint(EndpointName=os.environ["ENDPOINT_NAME"],
                                   ContentType="text/csv", Body=row)
    score = float(resp["Body"].read().decode().strip())
    print("score:", score)
    assert 0.0 <= score <= 1.0

The **buildspec** tells CodeBuild what to run: `install` (tools), `build` (the tests), and a test report in JUnit format.
A failing command fails the build, which stops the pipeline.

In [ ]:
%%writefile scripts/ci/buildspec.yml
version: 0.2
phases:
  install:
    runtime-versions:
      python: 3.11
    commands:
      - pip install -q pytest pandas numpy boto3
  build:
    commands:
      - echo "1. unit test"
      - python -m pytest -v --junitxml=reports/unit.xml tests/test_features.py
      - echo "2. model-quality gate"
      - python -m pytest -v -s --junitxml=reports/quality.xml tests/test_model_quality.py
      - echo "3. endpoint smoke test"
      - python -m pytest -v -s --junitxml=reports/smoke.xml tests/test_endpoint_smoke.py
reports:
  safedrive-tests:
    files:
      - "reports/*.xml"
    file-format: JUNITXML

## 3.3.2 Configure and troubleshoot CodeBuild, CodePipeline and friends

| Service | Job |
|---|---|
| **CodeCommit** | Git repositories in AWS (closed to new customers since 2024; know it as a possible source) |
| **CodeConnections** | Connection to GitHub, GitLab or Bitbucket. Created `PENDING`; someone must finish the handshake in the console |
| **CodeBuild** | Runs `buildspec.yml` in a container: tests, builds, Docker images. Pay per build minute |
| **CodeDeploy** | Deploys to EC2, ECS or Lambda with canary, linear or all-at-once and automatic rollback (`appspec.yml`) |
| **CodePipeline** | Orchestrates stages: Source → Build/Test → Approval → Deploy. Stores artifacts in S3 |

CodePipeline with an S3 source needs a **versioned** bucket. Create a small one for the source zip and the pipeline artifacts.

In [ ]:
CICD_BUCKET = f"safedrive-cicd-{U.account_id()}-{REGION}"
if not exists(s3.head_bucket, Bucket=CICD_BUCKET):
    extra = {} if REGION == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": REGION}}
    s3.create_bucket(Bucket=CICD_BUCKET, **extra)
s3.put_bucket_versioning(Bucket=CICD_BUCKET, VersioningConfiguration={"Status": "Enabled"})
s3.put_bucket_tagging(Bucket=CICD_BUCKET, Tagging={"TagSet": U.TAGS})
print(CICD_BUCKET, s3.get_bucket_versioning(Bucket=CICD_BUCKET)["Status"])

Zip the "repository" (buildspec at the root of the zip) and upload it. This plays the role of a `git push`.

In [ ]:
SOURCE_KEY = "source/safedrive-fraud.zip"
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("scripts/ci"):
        for name in files:
            if "__pycache__" not in root and ".pytest_cache" not in root:
                path = os.path.join(root, name)
                z.write(path, os.path.relpath(path, "scripts/ci"))
    print(z.namelist())
s3.put_object(Bucket=CICD_BUCKET, Key=SOURCE_KEY, Body=buf.getvalue())
print(f"s3://{CICD_BUCKET}/{SOURCE_KEY}")

### CodeBuild project
A project = source + environment (image, compute size, environment variables) + service role.
The tests get their settings from environment variables.

In [ ]:
cb = U.client("codebuild")
CB_TAGS = [{"key": t["Key"], "value": t["Value"]} for t in U.TAGS]
BUILD_ENV = {"type": "LINUX_CONTAINER", "image": "aws/codebuild/standard:7.0",
             "computeType": "BUILD_GENERAL1_SMALL",
             "environmentVariables": [{"name": "MODEL_PACKAGE_GROUP", "value": GROUP},
                                      {"name": "ENDPOINT_NAME", "value": ENDPOINT},
                                      {"name": "AUC_THRESHOLD", "value": "0.75"},
                                      {"name": "PIPELINE_NAME", "value": PIPELINE_NAME}]}
print([v["name"] for v in BUILD_ENV["environmentVariables"]])

Create the test project with the S3 zip as source.

In [ ]:
TEST_PROJECT = "safedrive-fraud-tests"
if not cb.batch_get_projects(names=[TEST_PROJECT])["projects"]:
    cb.create_project(name=TEST_PROJECT, serviceRole=ROLE, environment=BUILD_ENV, tags=CB_TAGS,
                      source={"type": "S3", "location": f"{CICD_BUCKET}/{SOURCE_KEY}"},
                      artifacts={"type": "NO_ARTIFACTS"})
cb.batch_get_projects(names=[TEST_PROJECT])["projects"][0]["source"]

Start a build and wait for it (about 2 minutes).

In [ ]:
build_id = cb.start_build(projectName=TEST_PROJECT)["build"]["id"]

def build_status():
    return cb.batch_get_builds(ids=[build_id])["builds"][0]["buildStatus"]

U.wait_until(build_status, ok=("SUCCEEDED",), bad=("FAILED", "FAULT", "STOPPED", "TIMED_OUT"),
             delay=15, label="build")

Look at the build phases. When a build fails, the failed phase and its message are the first thing to check.

In [ ]:
build = cb.batch_get_builds(ids=[build_id])["builds"][0]
pd.DataFrame(build["phases"])[["phaseType", "phaseStatus", "durationInSeconds"]]

Read the test results from the build log in CloudWatch Logs.

In [ ]:
logs = U.client("logs")
events = logs.get_log_events(logGroupName=build["logs"]["groupName"],
                             logStreamName=build["logs"]["streamName"], startFromHead=True)["events"]
for e in events:
    if any(word in e["message"] for word in ["PASSED", "FAILED", "passed", "failed", "AUC", "score:"]):
        print(e["message"].rstrip())

### CodePipeline: source → test → retrain
Projects used inside CodePipeline take their source and artifacts from the pipeline (`CODEPIPELINE` type).
Project 1 runs the same tests.

In [ ]:
TEST_PROJECT_CP = "safedrive-fraud-tests-pipeline"
if not cb.batch_get_projects(names=[TEST_PROJECT_CP])["projects"]:
    cb.create_project(name=TEST_PROJECT_CP, serviceRole=ROLE, environment=BUILD_ENV, tags=CB_TAGS,
                      source={"type": "CODEPIPELINE"}, artifacts={"type": "CODEPIPELINE"})
print(TEST_PROJECT_CP)

Project 2 starts the SageMaker pipeline. Its buildspec is written inline in the project instead of in the repository.

In [ ]:
START_PROJECT = "safedrive-fraud-start-pipeline"
START_SPEC = """version: 0.2
phases:
  build:
    commands:
      - aws sagemaker start-pipeline-execution --pipeline-name $PIPELINE_NAME --pipeline-execution-display-name codepipeline-$CODEBUILD_BUILD_NUMBER
"""
if not cb.batch_get_projects(names=[START_PROJECT])["projects"]:
    cb.create_project(name=START_PROJECT, serviceRole=ROLE, environment=BUILD_ENV, tags=CB_TAGS,
                      source={"type": "CODEPIPELINE", "buildspec": START_SPEC},
                      artifacts={"type": "CODEPIPELINE"})
print(START_PROJECT)

Define the three stages. Each action has a type (category + provider), a configuration, and input or output artifacts.

In [ ]:
def action(name, category, provider, configuration, **artifacts):
    return {"name": name, "configuration": configuration, **artifacts,
            "actionTypeId": {"category": category, "owner": "AWS", "provider": provider, "version": "1"}}

SRC = [{"name": "SourceOutput"}]
stages = [
    {"name": "Source", "actions": [action(
        "S3Source", "Source", "S3", outputArtifacts=SRC,
        configuration={"S3Bucket": CICD_BUCKET, "S3ObjectKey": SOURCE_KEY, "PollForSourceChanges": "false"})]},
    {"name": "Test", "actions": [action(
        "RunTests", "Test", "CodeBuild", {"ProjectName": TEST_PROJECT_CP}, inputArtifacts=SRC)]},
    {"name": "Retrain", "actions": [action(
        "StartSageMakerPipeline", "Build", "CodeBuild", {"ProjectName": START_PROJECT}, inputArtifacts=SRC)]}]
print([s["name"] for s in stages])

Create the pipeline. A new pipeline starts its first execution at once.

In [ ]:
cp = U.client("codepipeline")
CP_NAME = "safedrive-fraud-cicd"
if not exists(cp.get_pipeline, name=CP_NAME):
    cp.create_pipeline(
        pipeline={"name": CP_NAME, "roleArn": ROLE, "pipelineType": "V2", "stages": stages,
                  "artifactStore": {"type": "S3", "location": CICD_BUCKET}},
        tags=CB_TAGS)
else:
    cp.start_pipeline_execution(name=CP_NAME)
print(CP_NAME, "started")

Wait for the execution (about 4 minutes).

In [ ]:
def cp_status():
    runs = cp.list_pipeline_executions(pipelineName=CP_NAME, maxResults=1)["pipelineExecutionSummaries"]
    return runs[0]["status"] if runs else "Starting"

time.sleep(10)
U.wait_until(cp_status, ok=("Succeeded",), bad=("Failed", "Stopped", "Cancelled"),
             delay=20, timeout=1800, label="codepipeline")

Stage by stage result. A failed stage shows here first; the action's link leads to the CodeBuild log.

In [ ]:
stage_states = cp.get_pipeline_state(name=CP_NAME)["stageStates"]
pd.DataFrame([{"stage": s["stageName"], "status": s.get("latestExecution", {}).get("status"),
               "action": s["actionStates"][0]["actionName"],
               "summary": s["actionStates"][0].get("latestExecution", {}).get("summary", "")}
              for s in stage_states])

The last stage started a new execution of the SageMaker pipeline. Thanks to step caching it finishes in a few minutes
and registers a new model version.

In [ ]:
runs = sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)["PipelineExecutionSummaries"]
pd.DataFrame(runs)[["PipelineExecutionDisplayName", "PipelineExecutionStatus", "StartTime"]]

To release a change you upload a new zip and start the pipeline (with a Git source and CodeConnections this happens on every push):

```python
s3.put_object(Bucket=CICD_BUCKET, Key=SOURCE_KEY, Body=new_zip)
cp.start_pipeline_execution(name=CP_NAME)
```

**CodeConnections (note):** `codeconnections.create_connection(ProviderType="GitHub", ConnectionName="safedrive-github")`
returns a connection in `PENDING` status. A person must open it in the console and authorize the GitHub app. Until then
the source action fails. The pipeline role needs `codeconnections:UseConnection`. The source action then uses provider
`CodeStarSourceConnection` with `ConnectionArn`, `FullRepositoryId` and `BranchName`.

**CodeDeploy (note):** not used for SageMaker endpoints (SageMaker has its own deployment guardrails). Use it when the
model runs on Lambda, ECS or EC2: `appspec.yml` + a deployment configuration such as `CodeDeployDefault.LambdaCanary10Percent5Minutes`.

### Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| Source action fails: connection not available | CodeConnections connection is still `PENDING` | Finish the handshake in the console |
| Source action fails with S3 source | Bucket is not versioned, or wrong object key | Enable versioning; check `S3ObjectKey` |
| Build fails in `DOWNLOAD_SOURCE` | Service role cannot read the source or artifact bucket (or its KMS key) | Add S3 and KMS permissions to the CodeBuild role |
| `YAML_FILE_ERROR` / buildspec not found | `buildspec.yml` is not at the root of the source, or bad indentation | Move the file or set the buildspec path in the project |
| `AccessDenied` on `sagemaker:StartPipelineExecution` or `iam:PassRole` | CodeBuild role lacks permissions | Add the action; allow `iam:PassRole` for the SageMaker role |
| Pipeline cannot assume its role | Trust policy does not list `codepipeline.amazonaws.com` (or `codebuild`) | Fix the role trust policy |
| Build cannot reach PyPI or AWS APIs | Project runs in a VPC without NAT or VPC endpoints | Add NAT gateway or endpoints |
| Docker build fails in CodeBuild | Privileged mode is off | Set `privilegedMode: true` in the environment |
| Pipeline does not start on a new upload | `PollForSourceChanges` is false and no EventBridge rule | Add an EventBridge rule for the S3 object, or start it manually |

## 3.3.5 Mechanisms to retrain models

| Trigger | EventBridge rule | Example |
|---|---|---|
| Time | Schedule (`cron` or `rate`) | Retrain every Sunday night |
| New data | Event pattern on S3 `Object Created` | A new claims file lands in `incoming/` |
| Drift or quality drop | Event pattern on `CloudWatch Alarm State Change` | Model Monitor metric crosses a threshold (notebook 4.1) |
| Code change | CodePipeline | A push starts tests, then the pipeline (section above) |

- A SageMaker pipeline is a native EventBridge **target**. The rule needs a role that may call `sagemaker:StartPipelineExecution`.
- `SageMakerPipelineParameters` sets pipeline parameters for the triggered run.

Rule 1: a weekly schedule. We create it `DISABLED` so it does not start runs while you study.

In [ ]:
events = U.client("events")
RULE_SCHEDULE, RULE_S3, RULE_DRIFT = "safedrive-retrain-weekly", "safedrive-retrain-new-data", "safedrive-retrain-drift"
events.put_rule(Name=RULE_SCHEDULE, ScheduleExpression="cron(0 2 ? * SUN *)", State="DISABLED",
                Description="Retrain the fraud model every Sunday 02:00 UTC", Tags=U.TAGS)
events.put_targets(Rule=RULE_SCHEDULE, Targets=[{"Id": "fraud-pipeline", "Arn": pipeline_arn, "RoleArn": ROLE}])
events.describe_rule(Name=RULE_SCHEDULE)["ScheduleExpression"]

Rule 2: new data. S3 sends events to EventBridge only if the bucket has EventBridge notifications turned on.
This cell turns it on while keeping any other notification settings of the bucket.

In [ ]:
config = s3.get_bucket_notification_configuration(Bucket=BUCKET)
config.pop("ResponseMetadata", None)
config["EventBridgeConfiguration"] = {}
s3.put_bucket_notification_configuration(Bucket=BUCKET, NotificationConfiguration=config)
print("notification settings:", list(config))

The event pattern matches objects created under `safedrive/incoming/`. The target passes the new file to the pipeline as the `ClaimsData` parameter.

In [ ]:
INCOMING_KEY = "safedrive/incoming/claims.csv"
pattern = {"source": ["aws.s3"], "detail-type": ["Object Created"],
           "detail": {"bucket": {"name": [BUCKET]}, "object": {"key": [{"prefix": "safedrive/incoming/"}]}}}
events.put_rule(Name=RULE_S3, EventPattern=json.dumps(pattern), State="ENABLED", Tags=U.TAGS)
events.put_targets(Rule=RULE_S3, Targets=[{
    "Id": "fraud-pipeline", "Arn": pipeline_arn, "RoleArn": ROLE,
    "SageMakerPipelineParameters": {"PipelineParameterList": [
        {"Name": "ClaimsData", "Value": U.s3_uri(INCOMING_KEY)}]}}])
print(json.dumps(pattern))

Rule 3: drift. Model Monitor publishes CloudWatch metrics; an alarm on them changes state; this rule reacts to the
`ALARM` state of any alarm whose name starts with `safedrive-fraud-drift`. Created `DISABLED`.

In [ ]:
drift_pattern = {"source": ["aws.cloudwatch"], "detail-type": ["CloudWatch Alarm State Change"],
                 "detail": {"alarmName": [{"prefix": "safedrive-fraud-drift"}], "state": {"value": ["ALARM"]}}}
events.put_rule(Name=RULE_DRIFT, EventPattern=json.dumps(drift_pattern), State="DISABLED", Tags=U.TAGS)
events.put_targets(Rule=RULE_DRIFT, Targets=[{"Id": "fraud-pipeline", "Arn": pipeline_arn, "RoleArn": ROLE}])
pd.DataFrame(events.list_rules(NamePrefix="safedrive-retrain")["Rules"])[["Name", "State"]]

Test the new-data trigger: copy the claims file to the `incoming/` prefix, as if a new extract had arrived.

In [ ]:
before = len(sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)["PipelineExecutionSummaries"])
s3.copy_object(Bucket=BUCKET, Key=INCOMING_KEY, CopySource={"Bucket": BUCKET, "Key": f"{RAW}/claims.csv"})
print("executions before:", before)

Within a minute EventBridge starts a new pipeline execution. Wait until the number of executions goes up.

In [ ]:
def execution_count():
    return len(sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)["PipelineExecutionSummaries"])

U.wait_until(lambda: execution_count() > before, ok=(True,), delay=10, timeout=300, label="new execution started")
newest = sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)["PipelineExecutionSummaries"][0]
print(newest["PipelineExecutionArn"])

Check that the triggered run received the parameter from the rule.

In [ ]:
sm.list_pipeline_parameters_for_execution(PipelineExecutionArn=newest["PipelineExecutionArn"])["PipelineParameters"]

We only wanted to see the trigger fire. Stop this run (a full retrain would take another 15 minutes) and disable the rule.

In [ ]:
U.safe(sm.stop_pipeline_execution, PipelineExecutionArn=newest["PipelineExecutionArn"])
events.disable_rule(Name=RULE_S3)
print(events.describe_rule(Name=RULE_S3)["State"])

Other ways to start retraining (notes): a Lambda function that calls `start_pipeline_execution` (when you need custom logic
such as "only if 10,000 new rows"), Step Functions for multi-service workflows, and SageMaker Pipelines' own schedule
feature in Studio (it creates an EventBridge Scheduler schedule for you).

## Exam traps

- `ConditionStep` cannot read S3 by itself: it needs a `PropertyFile` + `JsonGet` (or a step property).
- Registering a model does not deploy it. Approval changes a status; something else (EventBridge, CodePipeline) must act on it.
- Blue/green needs capacity for a second fleet. When quota is tight, the answer is a rolling deployment.
- Auto-rollback needs CloudWatch **alarms** named in `AutoRollbackConfiguration`; without alarms nothing rolls back.
- A CodeConnections connection is unusable until its status is `AVAILABLE` (manual console step).

## Quiz

**1.** A pipeline must register a model only when the F1 score written by the evaluation step to `metrics.json` is above 0.8.
Which combination does this?
A. CallbackStep + SQS  B. PropertyFile + JsonGet + ConditionStep  C. LambdaStep + TransformStep  D. FailStep + CacheConfig
<details><summary>Answer</summary>B. The PropertyFile exposes the JSON output, JsonGet reads the value and the ConditionStep branches on it.</details>

**2.** A team updates a production endpoint with 10 instances. They want 10% of traffic on the new model for 10 minutes and an
automatic return to the old model if the 5XX error alarm fires. What should they configure?
A. Two variants with weights 90/10  B. Blue/green with canary traffic shifting and an auto-rollback alarm  C. A shadow variant  D. All-at-once with a long termination wait
<details><summary>Answer</summary>B. Canary shifting with a baking period and AutoRollbackConfiguration alarms does exactly this, with no manual steps.</details>

**3.** A CodePipeline with a GitHub source fails in the Source stage right after it was created with CloudFormation.
The error says the connection is not available. What is the cause?
A. The buildspec is missing  B. The artifact bucket is not versioned  C. The CodeConnections connection is still pending and needs a manual handshake  D. CodeBuild needs privileged mode
<details><summary>Answer</summary>C. Connections created by API or CloudFormation start as PENDING until someone completes the authorization in the console.</details>

**4.** The fraud model must be retrained whenever Model Monitor detects data drift. Which design needs the least custom code?
A. A cron job on EC2 that checks S3  B. CloudWatch alarm on the drift metric → EventBridge rule → SageMaker pipeline target  C. A notebook run by hand  D. CodeDeploy with a canary configuration
<details><summary>Answer</summary>B. All parts are managed: the alarm changes state, EventBridge matches the event and starts the pipeline directly.</details>

**5.** Auditors ask which data, code and metrics produced the model version that was live last March, and who approved it.
Which service keeps this record?
A. Amazon ECR image tags  B. SageMaker Model Registry (with lineage)  C. S3 versioning alone  D. CloudWatch Logs Insights
<details><summary>Answer</summary>B. Each model package version stores the artifact, metrics, approval status and approver, with lineage to the training job.</details>

## Save state

In [ ]:
saved = U.save_state(model_package_group=GROUP, pipeline_name=PIPELINE_NAME)
{k: saved[k] for k in ["model_package_group", "pipeline_name"]}

## Cleanup

**Kept for later notebooks:** the SageMaker pipeline `safedrive-fraud-pipeline`, the model package group `safedrive-fraud`
with its versions, and the serverless endpoint `safedrive-fraud` (none of them cost money while idle).

Delete the production endpoint, its configs and models (safety net; the endpoint was deleted above).

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Serverless endpoint `safedrive-fraud` | 4.2, 4.3a | Redeployed by the catch-up cell |
| Pipeline `safedrive-fraud-pipeline` | 4.1a (optional: target of the drift-to-retrain rule) | Not rebuilt: the rule is created without a target |
| Model package group `safedrive-fraud` | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=PROD)
for name in [BLUE, GREEN]:
    U.safe(sm.delete_endpoint_config, EndpointConfigName=name)
    U.safe(sm.delete_model, ModelName=name)
U.safe(cw.delete_alarms, AlarmNames=[ROLLBACK_ALARM])

Delete the EventBridge rules (targets first).

In [ ]:
for rule in [RULE_SCHEDULE, RULE_S3, RULE_DRIFT]:
    U.safe(events.remove_targets, Rule=rule, Ids=["fraud-pipeline"])
    U.safe(events.delete_rule, Name=rule)
U.safe(s3.delete_object, Bucket=BUCKET, Key=INCOMING_KEY)

Delete the CodePipeline and the CodeBuild projects.

In [ ]:
U.safe(cp.delete_pipeline, name=CP_NAME)
for project in [TEST_PROJECT, TEST_PROJECT_CP, START_PROJECT]:
    U.safe(cb.delete_project, name=project)

Delete the CodeBuild test report groups (created by the `reports` section of the buildspec).

In [ ]:
for arn in cb.list_report_groups()["reportGroups"]:
    if "safedrive-fraud-tests" in arn:
        U.safe(cb.delete_report_group, arn=arn, deleteReports=True)

Empty and delete the CI/CD bucket. A versioned bucket must have all object versions removed first.

In [ ]:
U.safe(U.session.resource("s3").Bucket(CICD_BUCKET).object_versions.delete)
U.safe(s3.delete_bucket, Bucket=CICD_BUCKET)

Stop any pipeline execution that is still running (for example the one started by CodePipeline).

In [ ]:
for run in sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)["PipelineExecutionSummaries"]:
    if run["PipelineExecutionStatus"] == "Executing":
        U.safe(sm.stop_pipeline_execution, PipelineExecutionArn=run["PipelineExecutionArn"])
print("done")